# Week 5: Advanced Ensemble Algorithms
## Notebook 11: Extremely Randomized Trees (ExtraTrees) from Scratch

### 1. Conceptual Intuition & Theoretical Background
In standard Random Forests, the best split threshold for each sampled feature is found by sorting values and picking the threshold that strictly maximizes Information Gain. This threshold search accounts for substantial computational overhead.

**Extremely Randomized Trees (ExtraTrees, Geurts et al., 2006):**
Introduces extreme randomization at two levels:
1. **Random Threshold Selection:** For each candidate feature, split thresholds are drawn **completely at random** from a uniform distribution $[\min(X_j), \max(X_j)]$, rather than testing all valid cuts.
2. **Full Dataset Usage:** ExtraTrees typically trains on the original whole dataset (without bootstrap bagging), relying purely on random splits to generate diversity.

**Bias-Variance Implication:**
- Slightly increases individual tree bias.
- Drastically reduces ensemble variance by decreasing correlation between individual trees:
$$\text{Var}(\bar{T}) = \rho \sigma^2 + \frac{1 - \rho}{M} \sigma^2$$
As inter-tree correlation $\rho \to 0$, ensemble variance decreases significantly.

---

### 2. Scratch Python Implementation: ExtraTrees Splitter

In [ ]:
import numpy as np

class ExtraTreeLeaf:
    def __init__(self, value):
        self.value = value

class ExtraTreeNode:
    def __init__(self, feature, threshold, left, right):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right

def build_extratree_scratch(X, y, current_depth=0, max_depth=6, min_samples_split=2):
    n_samples, n_features = X.shape
    
    # Stopping criteria
    if current_depth >= max_depth or n_samples < min_samples_split or len(np.unique(y)) == 1:
        return ExtraTreeLeaf(np.mean(y))
        
    # Subsample candidate features (e.g. sqrt(p))
    max_features = max(1, int(np.sqrt(n_features)))
    selected_features = np.random.choice(n_features, size=max_features, replace=False)
    
    best_feat = None
    best_thresh = None
    best_mse = np.inf
    
    # ExtraTrees core: Draw split thresholds uniformly at random!
    for f in selected_features:
        f_min, f_max = X[:, f].min(), X[:, f].max()
        if f_min == f_max:
            continue
            
        # Draw random split threshold
        thresh = np.random.uniform(f_min, f_max)
        
        left_mask = X[:, f] <= thresh
        right_mask = ~left_mask
        
        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
            continue
            
        mse = (np.var(y[left_mask]) * np.sum(left_mask) + 
               np.var(y[right_mask]) * np.sum(right_mask)) / n_samples
               
        if mse < best_mse:
            best_mse = mse
            best_feat = f
            best_thresh = thresh
            
    if best_feat is None:
        return ExtraTreeLeaf(np.mean(y))
        
    left_child = build_extratree_scratch(
        X[X[:, best_feat] <= best_thresh], y[X[:, best_feat] <= best_thresh],
        current_depth + 1, max_depth, min_samples_split
    )
    right_child = build_extratree_scratch(
        X[X[:, best_feat] > best_thresh], y[X[:, best_feat] > best_thresh],
        current_depth + 1, max_depth, min_samples_split
    )
    
    return ExtraTreeNode(best_feat, best_thresh, left_child, right_child)

# Synthetic Regression Benchmark
np.random.seed(42)
X_reg = np.random.uniform(-3, 3, (200, 5))
y_reg = X_reg[:, 0] ** 2 + np.sin(X_reg[:, 1]) + np.random.normal(0, 0.1, 200)

tree = build_extratree_scratch(X_reg, y_reg, max_depth=5)

def predict_extratree(node, x):
    if isinstance(node, ExtraTreeLeaf):
        return node.value
    if x[node.feature] <= node.threshold:
        return predict_extratree(node.left, x)
    return predict_extratree(node.right, x)

preds = [predict_extratree(tree, sample) for sample in X_reg[:5]]
print("--- ExtraTrees Random Threshold Fitting ---")
print(f"Sample Predictions:    {np.round(preds, 3)}")
print(f"Sample Ground Truth:   {np.round(y_reg[:5], 3)}")
print("Threshold search speedup achieved by skipping sorting loops entirely!")

### 3. Step-by-Step Code Walkthrough
1. **Random Threshold Draw:** Uses `np.random.uniform(f_min, f_max)` instead of sorting and testing all midpoints.
2. **Computational Speedup:** Reduces node splitting complexity from $O(N \log N)$ to $O(N)$.
3. **Variance Reduction:** Produces smoother decision boundaries by averaging over randomized cut points.

---

### 4. Industry Applications & Trade-offs
- **High-Dimensional Biological Genomics:** Fast baseline training across datasets with $>50,000$ genetic markers.
- **Trade-off:** Individual trees have higher bias than Random Forest trees.

---

### 5. Top Interview Questions & Answers
**Q1: Why is ExtraTrees significantly faster to train than Random Forest?**
> **Answer:** Random Forest sorts candidate feature values at every node to find the optimal split threshold ($O(N \log N)$). ExtraTrees draws a random threshold uniformly from the feature range ($O(1)$), evaluating the split in a single $O(N)$ pass.

**Q2: Why does ExtraTrees generally not use bootstrap sampling?**
> **Answer:** Random Forests rely on bootstrap aggregation (bagging) to create tree diversity. Because ExtraTrees introduces extreme randomization during threshold selection, it does not require bagging to achieve low inter-tree correlation.